In [1]:
!pip install -r ../requirements.txt

In [2]:
import gzip
import json
import os
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
def load_data(filepath):
    rows=[]
    with gzip.open(filepath, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

In [ ]:
raw_path = "../data/raw/modcloth_final_data.json.gz"
out_dir = "../data/processed"
os.makedirs(out_dir, exist_ok=True)

df = load_data(raw_path)
print(f"Loaded {len(df)} rows and {len(df.columns)} columns.")
print("Columns:", list(df.columns))
initial_rows=len(df)

# Drop any row where fit is empty
df = df.dropna(subset=["fit"]).copy()
after_drop = len(df)
print(f"Dropped {initial_rows - after_drop} rows with missing 'fit'.")

label_map = {"small": 0, "fit": 1, "large": 2}
df["fit_label"] = df["fit"].map(label_map)

# Drop any row where fit is not consistent
df = df.dropna(subset=["fit_label"]).copy()
after_mapping = len(df)
print(f"Dropped {after_drop - after_mapping} rows with unexpected 'fit' values.")
print(f"Total rows dropped: {initial_rows - after_mapping} ({after_mapping} remaining).")

df["fit_label"] = df["fit_label"].astype(int)

Loaded 82790 rows and 18 columns.
Columns: ['item_id', 'waist', 'size', 'quality', 'cup size', 'hips', 'bra size', 'category', 'bust', 'height', 'user_name', 'length', 'fit', 'user_id', 'shoe size', 'shoe width', 'review_summary', 'review_text']
Dropped 0 rows with missing 'fit'.
Dropped 0 rows with unexpected 'fit' values.
Total rows dropped: 0 (82790 remaining).


In [8]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["fit_label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["fit_label"],
)

train_df.to_csv(os.path.join(out_dir, "train.csv"), index=False)
val_df.to_csv(os.path.join(out_dir, "val.csv"), index=False)
test_df.to_csv(os.path.join(out_dir, "test.csv"), index=False)
print(f"Saved splits -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Saved splits -> Train: 66232, Val: 8279, Test: 8279
